# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [3]:
from peft import prepare_model_for_kbit_training

from generation.model import load_base_model
from generation.data import load_train_df
from generation.train import train_adapter
from generation.style_loss import build_style_weights, top_tokens

model, tokenizer = load_base_model()
model = prepare_model_for_kbit_training(model)

train_df = load_train_df()

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Main adapters (LoRA)

In [4]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8 (overwrite=True to force)


'artifacts/adapters/tool_lora_r8'

## Ablation: DoRA (same artist)

In [5]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=True)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=True)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_dora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_dora_r8 (overwrite=True to force)


'artifacts/adapters/tool_dora_r8'

## Ablation: Rank

In [6]:
train_adapter(model, tokenizer, train_df, "Gojira", r=4, use_dora=False)
train_adapter(model, tokenizer, train_df, "Gojira", r=16, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r4 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r16 (overwrite=True to force)


'artifacts/adapters/gojira_lora_r16'

## Remaining artists (LoRA + DoRA r=8)

Completes the lineup so the attribution table and perplexity diagonal cover all 5 artists, and the LoRA-vs-DoRA ablation generalizes beyond Gojira/Tool.

In [7]:
for artist in ["Death", "Meshuggah", "Opeth"]:
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=False, overwrite=False)
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=True, overwrite=False)

[skip] adapter exists, not retraining: artifacts/adapters/death_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/death_dora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/meshuggah_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/meshuggah_dora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/opeth_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/opeth_dora_r8 (overwrite=True to force)


## Experiment: Style-Weighted Loss

Up-weight artist-distinctive tokens (token-level TF-IDF) in the cross-entropy so
the adapter learns artist vocabulary, not just genre. Building/inspecting the
weights is CPU-only; only the `train_adapter` call below needs the GPU. Trains to
`*_sw` adapters so they can be A/B'd against the plain ones on attribution.

In [10]:
style_w_gojira = build_style_weights("Gojira", train_df, tokenizer, text_col="clean")
top_tokens(style_w_gojira, tokenizer, n=30)

  attention      5.5
  space          5.5
  forest         5.5
  !              5.5
  land           5.5
  strength       5.5
  respect        5.5
  …              5.5
  get            5.5
  New            5.5
  structure      5.5
  code           5.5
  stars          5.5
  fight          5.5
  forces         5.5
  understanding  5.5
  core           5.5
  dig            5.5
  cycle          5.5
  positive       5.5
  lock           5.5
  Get            5.5
  bow            5.5
  low            5.5
  load           5.5
  Enter          5.5
  trees          5.5
  bag            5.5
  ready          5.5
  create         5.5


[('attention', 5.5),
 ('space', 5.5),
 ('forest', 5.5),
 ('!', 5.5),
 ('land', 5.5),
 ('strength', 5.5),
 ('respect', 5.5),
 ('…', 5.5),
 ('get', 5.5),
 ('New', 5.5),
 ('structure', 5.5),
 ('code', 5.5),
 ('stars', 5.5),
 ('fight', 5.5),
 ('forces', 5.5),
 ('understanding', 5.5),
 ('core', 5.5),
 ('dig', 5.5),
 ('cycle', 5.5),
 ('positive', 5.5),
 ('lock', 5.5),
 ('Get', 5.5),
 ('bow', 5.5),
 ('low', 5.5),
 ('load', 5.5),
 ('Enter', 5.5),
 ('trees', 5.5),
 ('bag', 5.5),
 ('ready', 5.5),
 ('create', 5.5)]

In [11]:
style_w_tool = build_style_weights("Tool", train_df, tokenizer, text_col="clean")
top_tokens(style_w_tool, tokenizer, n=30)

  press       5.5
  shoving     5.5
  1           5.5
  finger      5.5
  hey         5.5
  swim        5.5
  Love        5.5
  Fuck        5.5
  .           5.5
  Dr          5.5
  cheat       5.5
  calling     5.5
  dumb        5.276
  necessary   5.276
  wanna       5.088
  tempest     4.977
  weather     4.679
  la          4.578
  Jesus       4.38
  frightened  4.38
  dick        4.082
  adds        4.082
  fucking     3.911
  Hey         3.729
  him         3.721
  two         3.559
  little      3.559
  shit        3.532
  8           3.485
  nard        3.485


[('press', 5.5),
 ('shoving', 5.5),
 ('1', 5.5),
 ('finger', 5.5),
 ('hey', 5.5),
 ('swim', 5.5),
 ('Love', 5.5),
 ('Fuck', 5.5),
 ('.', 5.5),
 ('Dr', 5.5),
 ('cheat', 5.5),
 ('calling', 5.5),
 ('dumb', 5.276),
 ('necessary', 5.276),
 ('wanna', 5.088),
 ('tempest', 4.977),
 ('weather', 4.679),
 ('la', 4.578),
 ('Jesus', 4.38),
 ('frightened', 4.38),
 ('dick', 4.082),
 ('adds', 4.082),
 ('fucking', 3.911),
 ('Hey', 3.729),
 ('him', 3.721),
 ('two', 3.559),
 ('little', 3.559),
 ('shit', 3.532),
 ('8', 3.485),
 ('nard', 3.485)]

In [12]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False, style_weights=style_w_gojira)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=False, style_weights=style_w_tool)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8_sw (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8_sw (overwrite=True to force)


'artifacts/adapters/tool_lora_r8_sw'